# Lesson 5 - Obstacle avoidance

This capstone combines the LIDAR and the motors. The robot drives forward while the
front is clear and turns toward the more open side when something is close.

**Safety: keep the robot on a clear floor, or lift the wheels to watch the logic
without moving.**

## The decision rule

For each scan we compute the nearest obstacle in four sectors:

- if the **front** sector is farther than 0.5 m, drive **forward**
- otherwise turn toward whichever of **left** / **right** is more open

In [ ]:
import sys
sys.path.insert(0, "..")
from jbot import Robot, lidar_sectors

robot = Robot()
sectors = lidar_sectors(max_scans=1)
front = sectors["front"]
if front is not None and front > 0.5:
    decision = "forward"
elif (sectors["left"] or 0) > (sectors["right"] or 0):
    decision = "left"
else:
    decision = "right"
print(sectors, "->", decision)

## Run the reactive loop

This keeps the LIDAR open and reacts continuously. The readout shows the current
action and sector distances. Press **STOP** to halt the robot and the loop.

In [ ]:
import threading
import ipywidgets as widgets
from IPython.display import display
from rplidar import RPLidar

out = widgets.HTML()
stop_btn = widgets.Button(description="STOP", button_style="danger")
display(out, stop_btn)
state = {"run": True}

def sectors_from_scan(scan):
    buckets = {"front": [], "right": [], "back": [], "left": []}
    for (_, angle, dist) in scan:
        if dist <= 0:
            continue
        if angle >= 315 or angle < 45:
            buckets["front"].append(dist)
        elif angle < 135:
            buckets["right"].append(dist)
        elif angle < 225:
            buckets["back"].append(dist)
        else:
            buckets["left"].append(dist)
    return {k: (min(v) / 1000.0 if v else None) for k, v in buckets.items()}

def worker():
    lidar = RPLidar("/dev/ttyUSB0", baudrate=115200)
    try:
        for scan in lidar.iter_scans(max_buf_meas=2000):
            if not state["run"]:
                break
            s = sectors_from_scan(scan)
            front = s["front"]
            if front is not None and front > 0.5:
                robot.forward(0.3)
                action = "forward"
            elif (s["left"] or 0) > (s["right"] or 0):
                robot.left_turn(0.3)
                action = "left"
            else:
                robot.right_turn(0.3)
                action = "right"
            out.value = "<b>%s</b><br>%s" % (action, s)
    finally:
        robot.stop()
        lidar.stop()
        lidar.stop_motor()
        lidar.disconnect()

stop_btn.on_click(lambda _: state.update(run=False))
threading.Thread(target=worker, daemon=True).start()